In [0]:
%pip install mlflow scikit-learn seaborn matplotlib --quiet
dbutils.library.restartPython()

In [0]:
import sys, os
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

In [0]:
src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)

In [0]:
# --- CELDA 3: IMPORTS ---
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import pandas as pd
from nombre_paquete.evaluation import metrics 


In [0]:

# --- CELDA 4: CARGAR EL MEJOR MODELO ---

RUN_ID = "a3d41e88423b4280aa5d84190c68b621" 

try:
    # Intento de cargar desde MLflow (Best Practice)
    model_uri = f"runs:/{RUN_ID}/model"
    print(f" Cargando modelo desde: {model_uri}")
    model = mlflow.pyfunc.load_model(model_uri)
    print(" Modelo cargado exitosamente.")
except:
    print(" No se pudo cargar desde MLflow (¿Pusiste el ID?).")
    print("   -> Creando un modelo Dummy (RandomForest) para que el notebook no falle ahora.")
    from sklearn.ensemble import RandomForestRegressor
    model = RandomForestRegressor().fit([[0]], [0]) # Dummy


In [0]:

# --- CELDA 5: CARGAR DATOS DE TEST ---
# Leemos la tabla Silver de Test que guardamos en Preprocessing
# Esta tabla el modelo NUNCA la ha visto durante el entrenamiento.
print(" Cargando datos de Test (Silver Layer)...")
df_test = spark.table("climate_test_silver").toPandas()

TARGET = 'energy_consumption'
cols_to_drop = [TARGET, 'index', 'date', 'level_0']

X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])
y_test = df_test[TARGET]

# Asegurar que el dummy funcione si falló MLflow (Solo para demo)
if hasattr(model, "n_features_in_") and model.n_features_in_ == 1:
    model.fit(X_test, y_test) 

print(f"Datos de Test listos: {X_test.shape}")


In [0]:

# --- CELDA 6: GENERAR PREDICCIONES ---
print(" Generando predicciones...")
y_pred = model.predict(X_test)

# --- CELDA 7: CÁLCULO DE MÉTRICAS ---
resultados = metrics.calculate_metrics(y_test, y_pred)
metrics.print_metrics(resultados, model_name="Mejor Modelo (MLflow)")


In [0]:

# --- CELDA 8: VISUALIZACIÓN DE RESULTADOS ---

# 1. Real vs Predicho (Serie de tiempo)
# Muestra si el modelo captura la tendencia general
metrics.plot_real_vs_predicted(y_test, y_pred, limit=150, title="Performance en Test: Real vs Predicho")

# 2. Análisis de Residuales (Diagnóstico profundo)
# - Si la nube de puntos no tiene forma (es aleatoria), el modelo es bueno.
# - Si ves una curva ("U" o línea), faltan variables o el modelo es muy simple.
metrics.plot_residuals(y_test, y_pred)

In [0]:

# 1. Definir nombre del modelo en el Registro
# Esto crea una entrada única en el "Model Registry" de Databricks
registry_name = "Climate_Energy_Predictor"

# 2. Registrar el mejor modelo (Usando el RUN_ID que ya tenías cargado)
print(f"®️ Registrando modelo '{registry_name}' desde Run ID: {RUN_ID}")


model_uri = f"runs:/{RUN_ID}/model"
model_details = mlflow.register_model(model_uri=model_uri, name=registry_name)

# 3. Transicionar a "Production" (Opcional pero recomendado)
# Esto le dice al sistema: "Esta es la versión oficial que debe usar el negocio"
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.transition_model_version_stage(
    name=registry_name,
    version=model_details.version,
    stage="Production",
    archive_existing_versions=True # Mueve el anterior production a archivado
)

print(f" Modelo {registry_name} (v{model_details.version}) promovido a PRODUCTION.")